# PINN 2D corregido y mejorado

Revisión y ampliación de septiembre de 2026 del notebook original de John Meyor Medina Diaz. El original se conserva sin cambios como evidencia histórica. Esta versión documenta una implementación corregida, experimentos nuevos y sus limitaciones; no se presenta como un método PINN nuevo ni como validación biológica.

El original se conserva como `fixtures/original_PINN_2D_Biology.ipynb` solo para auditoría estática, nunca se ejecuta. Este notebook utiliza el experimento compacto y evaluable de `pinn2d.py`; sus resultados no se atribuyen al entrenamiento histórico.

## Problema y corrección

Resolver $c_{xx}+c_{yy}-0.1c=0$ en $(0,1)^2$, con $c=1$ en el lado izquierdo abierto y $c=0$ en los otros tres lados abiertos. Las dos esquinas izquierdas tienen datos discontinuos; no se suavizan para facilitar el problema.

En DeepXDE, las derivadas correctas usan `(i=0,j=0)` y `(i=1,j=1)`. El original omitía `j=1` y calculaba una derivada mixta.

Usar un kernel con las dependencias de `requirements_cpu.txt`. No se instalan paquetes ni se cambia la configuración global desde el notebook.

In [ ]:
import os
os.environ['DDE_BACKEND'] = 'pytorch'
from pathlib import Path
import json
import inspect
import sys
import subprocess
from IPython.display import display, Image
from pinn2d import pde, boundary_lift
from run_experiment import audit_run
print(inspect.getsource(pde))


## Mejora matemática de las condiciones de borde

Se compara una PINN corregida con bordes penalizados (`soft`) contra una con una función analítica que representa los límites de las esquinas (`lifted`). La segunda aprende una corrección que vale cero en los cuatro lados.

La función analítica **no es la solución exacta**; sus valores no se ajustan a la serie de referencia. Ver la derivación y los criterios fijados antes de entrenar en `PROTOCOL.md`.

No basta mirar la pérdida: medimos error de solución, residuo independiente, errores de borde, comportamiento cerca de las esquinas y rango de concentración.

In [ ]:
print(inspect.getsource(boundary_lift))


## Reproducir un entrenamiento nuevo

El entrenamiento está desactivado por defecto. Cambia `RUN_NEW` a `True` y usa una carpeta nueva, o ejecuta `python execute_notebook.py --train --output training_executed.ipynb`. No se sobrescriben experimentos existentes. Para la comparación completa: `python run_experiment.py --output results_new`, con tres semillas por método.

Los resultados pueden no pasar todos los criterios: conserva todos los casos.

In [ ]:
RUN_NEW = os.environ.get('PINN_RUN_NEW', '0') == '1'
new_root = Path('results_notebook_run')
if RUN_NEW:
    subprocess.run([sys.executable, 'run_experiment.py', '--output', str(new_root),
                    '--modes', 'lifted', '--seeds', '0'], check=True)
    audit_run(new_root / 'lifted_seed0')
else:
    print('No new training requested; the next cells inspect the recorded experiments.')


## Inspeccionar los resultados medidos disponibles

La figura de semilla0 está preseleccionada, no elegida por ser el mejor resultado. La tabla muestra todas las semillas del experimento archivado. Se recargan los seis pesos y se contrastan sus predicciones con las guardadas, sin reentrenarlos. Los resultados son de mallas finitas y no prueban validación biológica o cotas de error continuas.

In [ ]:
root = Path('results_neural_v1')
summary_file = root / 'summary.json'
if summary_file.exists():
    rows = json.loads(summary_file.read_text(encoding='utf-8'))
    control = json.loads((root / 'analytic_lifting_only' / 'metrics.json').read_text(encoding='utf-8'))
    print('Analytical lifting only: relative_L2(%) =', 100*control['interior']['relative_l2'],
          'PDE_RMSE =', control['pde_test_rmse'])
    print('mode seed relative_L2(%) max_error PDE_RMSE pass')
    for row in rows:
        print(row['mode'], row['seed'],
              round(100*row['interior']['relative_l2'], 6),
              round(row['interior']['max_absolute'], 6),
              round(row['pde_test_rmse'], 6), row['all_criteria_pass'])
        report = audit_run(root / (row['mode'] + '_seed' + str(row['seed'])))
        assert report['audit_pass'], 'Checkpoint reload failed'
    print('All recorded checkpoint reloads passed.')
    figure = root / 'lifted_seed0' / 'comparison.png'
    if figure.exists():
        display(Image(filename=str(figure)))
else:
    raise FileNotFoundError('The recorded results_neural_v1 folder is required for this walkthrough.')
